# 🍪 Operation: Macaroon — A Cryptographic Access Control Lab

---

> *"Cryptography is not about hiding information. It's about making trust transferable."*

---

## Your Mission Briefing

Welcome, Agent.

You've been hired as a security consultant for **Galactic Industries HQ** — a gleaming skyscraper with 40 floors of offices, server rooms, break rooms, and executive suites. The building uses a cutting-edge cryptographic key system called **Macaroons** to manage access.

The CEO just left for a two-week sailing trip and handed her **Master Key** to the Head of Security before leaving. The Head of Security needs to grant access to various employees and contractors — *but the CEO cannot be reached, and no one should ever see the Master Key again*.

How does the system let you grant a Floor Manager access to Floor 1, and then let *that* Floor Manager hand a guest a pass to just the break room — **all without copying or exposing the master key?**

That's exactly what you're going to build today.

Over the course of this lab, you will:

1. **Build** the cryptographic primitive (HMAC) that makes Macaroons possible
2. **Mint** a root Macaroon using a master secret
3. **Attenuate** it — shaving off privileges — without the original secret
4. **Verify** a chain of trust end-to-end
5. **Attack** your own system to prove it holds

Let's begin.


---

## 🔐 Stage 1: The Vault's Primitive — HMAC

### Background: Why Not Just `Hash(Key + Message)`?

Before you can build a Macaroon, you need to understand the cryptographic lock at its heart: **HMAC** (Hash-based Message Authentication Code).

Your first instinct might be: *"I'll just hash the secret key together with the message!"* Something like:

```
signature = SHA256(master_secret + token_id)
```

This seems reasonable. But it has a catastrophic flaw called a **Length Extension Attack**.

SHA-256 works by processing data in 64-byte blocks, updating an internal state as it goes. When it outputs a digest, that output **is** the internal state. An attacker who sees `SHA256(secret + message)` can compute `SHA256(secret + message + evil_extension)` without ever knowing `secret`. They just feed the known digest back in as a starting state and keep hashing!

For an access control system, this would be catastrophic — an attacker could extend a legitimate token with `admin=true` and produce a valid-looking signature.

### The Real HMAC

HMAC defeats this by applying the key **twice**, using XOR-padding to create two independent keyed layers:

$$\text{HMAC}(K, M) = H\big((K \oplus opad) \;||\; H((K \oplus ipad) \;||\; M)\big)$$

Where:
- $K$ is the key, padded to the block size (64 bytes for SHA-256)
- $opad$ = `0x5c5c5c...` (outer padding, 64 bytes)
- $ipad$ = `0x363636...` (inner padding, 64 bytes)
- $||$ means concatenation
- $\oplus$ means XOR

The inner hash protects the message. The outer hash protects the inner hash with a *different* key transformation. Length extension attacks fail because an attacker would need to extend through both layers simultaneously — which is computationally infeasible.

### Road Map for Stage 1

Rather than implementing HMAC all at once, you'll build it **one step at a time**, test each step independently, and then assemble the pieces at the end. Here's the plan:

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `prepare_key` | Normalise the key to exactly 64 bytes |
| 2 | `make_padded_keys` | Produce the inner and outer XOR-padded versions |
| 3 | `inner_hash` | Hash the padded key + message |
| 4 | `outer_hash` | Hash the outer padded key + inner digest |
| ✦ | `manual_hmac` | Compose all four steps into the full HMAC |


In [ ]:
import hashlib
import hmac

BLOCK_SIZE = 64  # SHA-256 processes data in 64-byte (512-bit) blocks

---

### Step 1 of 4 — `prepare_key`: Normalise the Key

HMAC requires its key to be **exactly one block long** (64 bytes for SHA-256). Real-world keys come in all sizes, so we need a deterministic normalisation rule:

- **Too long (> 64 bytes)?** Hash it with SHA-256. The digest is 32 bytes, which is less than 64, so we then pad it.
- **Too short (< 64 bytes)?** Right-pad with null bytes (`\x00`) until it's exactly 64 bytes.
- **Exactly 64 bytes?** Leave it alone.

This guarantees that no matter what key a caller passes in, the rest of the algorithm always sees a predictable 64-byte value.

```
  raw key (any length)
       │
       ├─── len > 64 ──▶  SHA256(key)  ──▶  pad to 64
       ├─── len < 64 ──────────────────▶  pad to 64
       └─── len = 64 ──────────────────▶  use as-is
```

In [ ]:
def prepare_key(key: bytes) -> bytes:
    """
    Normalise an HMAC key to exactly BLOCK_SIZE (64) bytes.

    Rules:
      - If len(key) > BLOCK_SIZE: replace key with SHA256(key)
      - If len(key) < BLOCK_SIZE: right-pad with b'\x00' bytes
      - If len(key) == BLOCK_SIZE: return unchanged

    Args:
        key: The raw secret key of any length.

    Returns:
        A bytes object of length exactly BLOCK_SIZE.
    """
    # YOUR CODE HERE
    # Hint: hashlib.sha256(key).digest() compresses a long key to 32 bytes
    # Hint: key + b'\x00' * N appends N null bytes
    raise NotImplementedError("Implement prepare_key")

In [ ]:
# --- Tests for Step 1 ---

# Test A: a short key gets padded to 64 bytes
short_key = b'tiny'
result_short = prepare_key(short_key)
assert len(result_short) == BLOCK_SIZE, f"Expected 64 bytes, got {len(result_short)}"
assert result_short[:4] == b'tiny', "Original bytes should be preserved at the start"
assert result_short[4:] == b'\x00' * 60, "Remainder should be null bytes"
print(f"Short key padded:  {result_short.hex()[:24]}... (length={len(result_short)})")

# Test B: a key that is exactly 64 bytes passes through unchanged
exact_key = b'A' * 64
result_exact = prepare_key(exact_key)
assert result_exact == exact_key, "A 64-byte key should be returned unchanged"
print(f"Exact key unchanged: {'A' * 24}... (length={len(result_exact)})")

# Test C: a long key gets hashed, then padded to 64 bytes
long_key = b'B' * 128
result_long = prepare_key(long_key)
assert len(result_long) == BLOCK_SIZE, f"Expected 64 bytes after hashing, got {len(result_long)}"
expected_hashed = hashlib.sha256(long_key).digest()  # 32 bytes
assert result_long == expected_hashed + b'\x00' * 32, "Long key should be SHA256'd then padded"
print(f"Long key hashed+padded: {result_long.hex()[:24]}... (length={len(result_long)})")

print("\n✅ Step 1 passed: prepare_key is correct.")

---

### Step 2 of 4 — `make_padded_keys`: XOR the Key with opad and ipad

Now that the key is a predictable 64 bytes, we derive **two** different keys from it by XOR-ing every byte with a constant:

- **Inner padded key** (`i_key_pad`): XOR each byte with `0x36` (binary `00110110`)
- **Outer padded key** (`o_key_pad`): XOR each byte with `0x5c` (binary `01011100`)

The choice of `0x36` and `0x5c` is specified in [RFC 2104](https://www.rfc-editor.org/rfc/rfc2104). They're not magic — they just need to be different constants so the two padded keys are always distinct, even if the input key is all zeros.

XOR is its own inverse: `key ^ 0x36 ^ 0x36 == key`. So if you apply the same constant twice, you get back the original key. This also means the two padded keys are easy to compute but not trivially related to each other in a way an attacker can exploit.

```
prepared_key  ──XOR 0x36──▶  i_key_pad  (used for inner hash)
prepared_key  ──XOR 0x5c──▶  o_key_pad  (used for outer hash)
```

In [ ]:
def make_padded_keys(prepared_key: bytes) -> tuple[bytes, bytes]:
    """
    Derive the inner and outer XOR-padded keys from a normalised key.

    Args:
        prepared_key: A bytes object of exactly BLOCK_SIZE (64) bytes,
                      as returned by prepare_key().

    Returns:
        (i_key_pad, o_key_pad) — both are BLOCK_SIZE bytes long.
        i_key_pad: prepared_key XOR'd byte-by-byte with 0x36
        o_key_pad: prepared_key XOR'd byte-by-byte with 0x5c
    """
    # YOUR CODE HERE
    # Hint: bytes(x ^ 0x36 for x in prepared_key) builds the XOR'd byte string
    raise NotImplementedError("Implement make_padded_keys")

In [ ]:
# --- Tests for Step 2 ---

# Use an all-zero key so the XOR result is just the constant, byte-repeated
zero_key = b'\x00' * BLOCK_SIZE
i_pad, o_pad = make_padded_keys(zero_key)

assert len(i_pad) == BLOCK_SIZE, "i_key_pad must be BLOCK_SIZE bytes"
assert len(o_pad) == BLOCK_SIZE, "o_key_pad must be BLOCK_SIZE bytes"
assert i_pad == bytes([0x36] * BLOCK_SIZE), "XOR with all-zeros should give just the constant"
assert o_pad == bytes([0x5c] * BLOCK_SIZE), "XOR with all-zeros should give just the constant"
print(f"i_key_pad (first 8 bytes): {i_pad[:8].hex()}  (expect 3636363636363636)")
print(f"o_key_pad (first 8 bytes): {o_pad[:8].hex()}  (expect 5c5c5c5c5c5c5c5c)")

# XOR is its own inverse — applying it twice returns the original
recovered = bytes(x ^ 0x36 for x in i_pad)
assert recovered == zero_key, "Double XOR should recover the original key"
print(f"Double-XOR recovery:       {recovered[:8].hex()}  (expect 0000000000000000)")

# The two pads must be distinct
assert i_pad != o_pad, "i_key_pad and o_key_pad must differ"
print()
print("✅ Step 2 passed: make_padded_keys is correct.")

---

### Step 3 of 4 — `inner_hash`: The First Layer of Hashing

With `i_key_pad` in hand, we can now compute the **inner hash** — the first of the two SHA-256 calls:

```
inner_digest = SHA256( i_key_pad || message )
```

The `||` means concatenation: we prepend the 64-byte padded key directly in front of the message and feed the whole thing into SHA-256.

This is the layer that binds the message to the key. Because `i_key_pad` is derived from the secret, only someone who knows the secret can reproduce this digest.

The output is a 32-byte SHA-256 digest. This will be used as the *message* for the outer hash in Step 4.

In [ ]:
def inner_hash(i_key_pad: bytes, message: bytes) -> bytes:
    """
    Compute the inner HMAC hash: SHA256(i_key_pad || message).

    Args:
        i_key_pad: The inner XOR-padded key (BLOCK_SIZE bytes).
        message:   The original message to authenticate.

    Returns:
        A 32-byte SHA-256 digest.
    """
    # YOUR CODE HERE
    # Hint: concatenate with + and hash with hashlib.sha256(...).digest()
    raise NotImplementedError("Implement inner_hash")

In [ ]:
# --- Tests for Step 3 ---

test_i_key_pad = bytes([0x36] * BLOCK_SIZE)  # Simplified pad for testing
test_message   = b'Hello Macaroons'

result = inner_hash(test_i_key_pad, test_message)

# Output must be exactly 32 bytes (SHA-256 digest size)
assert len(result) == 32, f"Expected 32 bytes, got {len(result)}"

# Must match a direct SHA256 call with the same inputs concatenated
expected = hashlib.sha256(test_i_key_pad + test_message).digest()
assert result == expected, "inner_hash must equal SHA256(i_key_pad + message)"

# Changing the message must produce a different digest
different = inner_hash(test_i_key_pad, b'Different message')
assert different != result, "Different messages must produce different inner digests"

print(f"Inner digest: {result.hex()}")
print(f"Length:       {len(result)} bytes")
print()
print("✅ Step 3 passed: inner_hash is correct.")

---

### Step 4 of 4 — `outer_hash`: The Second Layer of Hashing

The outer hash wraps the inner digest in a second keyed layer:

```
final_digest = SHA256( o_key_pad || inner_digest )
```

This is what defeats the length extension attack. An attacker who intercepts `inner_digest` cannot extend it because it's immediately re-hashed with a *different* padded key (`o_key_pad`). To forge a new inner hash, they'd need to invert SHA-256 — which is computationally infeasible.

The result is the final 32-byte HMAC-SHA256 digest. This is the value that will serve as our Macaroon signature.

In [ ]:
def outer_hash(o_key_pad: bytes, inner_digest: bytes) -> bytes:
    """
    Compute the outer HMAC hash: SHA256(o_key_pad || inner_digest).

    Args:
        o_key_pad:    The outer XOR-padded key (BLOCK_SIZE bytes).
        inner_digest: The 32-byte result of inner_hash().

    Returns:
        The final 32-byte HMAC-SHA256 digest.
    """
    # YOUR CODE HERE
    # Same pattern as inner_hash — concatenate and hash
    raise NotImplementedError("Implement outer_hash")

In [ ]:
# --- Tests for Step 4 ---

test_o_key_pad   = bytes([0x5c] * BLOCK_SIZE)  # Simplified pad for testing
test_inner_digest = b'\xab' * 32              # Fake 32-byte inner digest

result = outer_hash(test_o_key_pad, test_inner_digest)

# Must be exactly 32 bytes
assert len(result) == 32, f"Expected 32 bytes, got {len(result)}"

# Must match a direct SHA256 call
expected = hashlib.sha256(test_o_key_pad + test_inner_digest).digest()
assert result == expected, "outer_hash must equal SHA256(o_key_pad + inner_digest)"

# The outer hash must differ from the inner digest it was given
assert result != test_inner_digest, "Outer hash should differ from its input"

# Swapping o_key_pad and i_key_pad should produce a different result
wrong_pad_result = outer_hash(bytes([0x36] * BLOCK_SIZE), test_inner_digest)
assert wrong_pad_result != result, "Using i_key_pad instead of o_key_pad must give a different result"

print(f"Outer digest: {result.hex()}")
print(f"Length:       {len(result)} bytes")
print()
print("✅ Step 4 passed: outer_hash is correct.")

---

### ✦ Assembly — `manual_hmac`: Composing the Four Steps

All four building blocks are ready. Now assemble them into the complete HMAC function.

The call sequence is exactly the pipeline you've been building:

```
raw key  ──prepare_key──▶  prepared_key
                           ──make_padded_keys──▶  i_key_pad, o_key_pad
                                                  ──inner_hash(i_key_pad, message)──▶  inner_digest
                                                  ──outer_hash(o_key_pad, inner_digest)──▶  final digest
```

At the end, `manual_hmac` must produce output that is **byte-for-byte identical** to Python's own `hmac` module. If it does, your implementation of HMAC is cryptographically correct and ready to power every stage that follows.

In [ ]:
def manual_hmac(key: bytes, message: bytes) -> bytes:
    """
    Compute HMAC-SHA256 by composing the four helper functions.

    HMAC(K, M) = outer_hash( o_key_pad, inner_hash( i_key_pad, M ) )
    where i_key_pad, o_key_pad = make_padded_keys( prepare_key(K) )

    Args:
        key:     The secret key (any length).
        message: The message to authenticate.

    Returns:
        A 32-byte HMAC-SHA256 digest.
    """
    # YOUR CODE HERE — call your four helper functions in order
    raise NotImplementedError("Compose the four steps into manual_hmac")

In [ ]:
# --- Final Verification: Match Python's hmac module ---

test_cases = [
    (b'short',              b'Hello Macaroons'),
    (b'exactly-64-bytes!!' + b'x' * 46, b'block-sized key test'),
    (b'a_very_long_key_' * 10,          b'long key test'),
    (b'super_secret_key', b''),          # Empty message
]

print("Running all test cases against Python's hmac module...\n")
for i, (k, m) in enumerate(test_cases, 1):
    yours    = manual_hmac(k, m)
    official = hmac.new(k, m, hashlib.sha256).digest()
    match    = "✅" if yours == official else "❌"
    label    = f"key={k[:12]}{'...' if len(k)>12 else ''!r}, msg={m[:16]!r}{'...' if len(m)>16 else ''}"
    print(f"  Case {i} {match}  {label}")
    assert yours == official, f"Mismatch on case {i}! Check your XOR constants and concatenation order."

print("\n✅ Stage 1 Complete: All cases match. The vault primitive is ready.")

---

## 🏛️ Stage 2: The Mint — Forging the Root Macaroon

### Background: What Is a Macaroon, Exactly?

A Macaroon (named after the mathematical concept, not the cookie — though the cookie is named after the same word) is a **bearer token with embedded caveats**. Unlike a simple API key, a Macaroon is a chain of HMAC signatures that proves a token was derived from a specific root secret without ever exposing that secret.

Think of it like a wax seal on a letter. The CEO seals a letter with her ring. The Head of Security can stamp *his* seal on top of hers — the final token carries both seals, and anyone with the original wax seal mold can verify the whole chain.

### The Root Macaroon

The CEO's **Master Secret** never leaves the vault. But it can produce a **Root Signature**:

```
Signature₀ = HMAC(master_secret, token_id)
```

Here, `token_id` is a human-readable label for this key (e.g., `"Manager-Key-001"`). It acts as a public identifier — the verifier will need it later to reproduce the chain. The signature is *secret*; the token_id is not.

This `Signature₀` is the Head of Security's master credential. He can hand it to employees — but he can also *attenuate* it before doing so.

### 🛠️ Your Task

Implement `mint_macaroon(token_id, master_secret)` which produces `Signature₀` by running HMAC with the `master_secret` as the key and the `token_id` (encoded as UTF-8 bytes) as the message.

In [ ]:
def mint_macaroon(token_id: str, master_secret: bytes) -> bytes:
    """
    Create the root Macaroon signature.
    
    This is the foundational operation: binding a token identifier
    to a master secret using HMAC. The output is Signature₀.
    
    Args:
        token_id:      A human-readable label for this root token (e.g., 'Manager-Key-001')
        master_secret: The CEO's vault secret — never shared, never exposed
    
    Returns:
        Signature₀: a 32-byte root HMAC digest
    """
    # YOUR CODE HERE
    # Hint: use your manual_hmac. The key is master_secret, the message is token_id as bytes.
    raise NotImplementedError("Implement mint_macaroon")

In [ ]:
# --- Mint the Root Token ---
master_secret = b'this_is_the_backend_secret'  # Locked in the CEO's vault
token_id      = 'Manager-Key-001'               # Public label; safe to share

sig0 = mint_macaroon(token_id, master_secret)

print(f"Token ID:               {token_id}")
print(f"Root Signature (sig0):  {sig0.hex()}")
print()
print("The Head of Security now holds sig0. The CEO is already on the boat.")
print("\n✅ Stage 2 Complete: Root Macaroon minted.")

---

## 🏢 Stage 3: Attenuation — The Floor Manager Gets a Restricted Pass

### Background: Delegating Without Sharing

Here is the magic at the heart of Macaroons.

The Head of Security wants to give the Floor 1 Manager access — but **only** to Floor 1. He can't hand over `sig0` directly, because then the Manager would have master-level access. And he can't call the CEO. So what does he do?

He uses `sig0` itself as a new HMAC key, and signs a **caveat** with it:

```
Signature₁ = HMAC(sig0, "floor=1")
```

He then gives the Manager **two things**:
- The list of caveats: `["floor=1"]`
- The final signature: `sig1`

Crucially:
- **The Manager cannot reverse `sig1` to get `sig0`** — HMAC is a one-way function.
- **The Manager cannot remove `floor=1`** — without `sig0` as the key, they can't reproduce `sig1` without the caveat.
- **The Manager cannot add new caveats freely** — though interestingly, they *can* add further restrictions (more on this in Stage 4).

This is called **attenuation**: you can only ever make a token *less* powerful, never more.

### The Chain So Far

```
master_secret ──HMAC──▶ sig0  ("Manager-Key-001")
    sig0      ──HMAC──▶ sig1  ("floor=1")
```

### 🛠️ Your Task

Implement `add_caveat(previous_signature, caveat)` which takes the previous signature as the HMAC key and the caveat string as the message, producing the next signature in the chain.

In [ ]:
def add_caveat(previous_signature: bytes, caveat: str) -> bytes:
    """
    Attenuate a Macaroon by appending a caveat.
    
    This uses the previous signature as the HMAC key, binding the new
    caveat into the chain. The previous signature is effectively 'consumed'
    — the caller should only pass the new signature onward.
    
    Args:
        previous_signature: The last signature in the chain (32 bytes)
        caveat:             A restriction string, e.g. 'floor=1' or 'expires=2024-12-31'
    
    Returns:
        The next signature in the chain (32 bytes)
    """
    # YOUR CODE HERE
    # Hint: HMAC where key=previous_signature and message=caveat (as UTF-8 bytes)
    raise NotImplementedError("Implement add_caveat")

In [ ]:
# --- Attenuate the Token: Floor Manager ---
caveat1 = 'floor=1'
sig1 = add_caveat(sig0, caveat1)

print("The Head of Security attenuates the master token...")
print()
print(f"Caveat applied:          '{caveat1}'")
print(f"Previous signature (hidden from Manager): {sig0.hex()[:16]}...")
print(f"New signature (sig1):    {sig1.hex()}")
print()
print("The Manager receives: token_id='Manager-Key-001', caveats=['floor=1'], sig1")
print("The Manager does NOT receive: sig0 or master_secret.")
print("\n✅ Stage 3 Complete: Token attenuated for the Floor Manager.")

---

## 🚪 Stage 4: Deeper Attenuation — The Guest Pass

### Background: Delegation All the Way Down

Now here's where it gets interesting. The Floor Manager wants to let their visiting friend into the building — but only to the break room on Floor 1. The Manager doesn't have `sig0` or `master_secret`. All they have is:
- `sig1` (their own attenuated token)
- Their list of caveats so far: `["floor=1"]`

Can they create a *further* restricted pass? **Yes!** That's the whole point.

The Manager runs:

```
Signature₂ = HMAC(sig1, "room=breakroom")
```

They give the friend:
- Caveats: `["floor=1", "room=breakroom"]`
- Signature: `sig2`

Note that the friend's token is **strictly less powerful** than the Manager's. The friend cannot access any room other than the break room. They cannot strip out `floor=1` to roam the whole floor. And they had no part in creating `sig0` or `sig1`.

### The Full Chain

```
master_secret ──HMAC──▶ sig0  (token_id: "Manager-Key-001")
    sig0      ──HMAC──▶ sig1  (caveat: "floor=1")
    sig1      ──HMAC──▶ sig2  (caveat: "room=breakroom")
```

### 🛠️ Your Task

Use `add_caveat` to append a second caveat (`room=breakroom`) to the chain. No new function needed — just show the chain extending naturally.

In [ ]:
# --- Attenuate Further: The Guest Pass ---
caveat2 = 'room=breakroom'

# YOUR CODE HERE
# Call add_caveat with sig1 and caveat2 to produce sig2
sig2 = None  # Replace this line
raise NotImplementedError("Compute sig2 using add_caveat")

In [ ]:
# --- Print the full chain for inspection ---
print("=== The Full Macaroon Chain ===")
print()
print(f"  master_secret  ──HMAC──▶  sig0  | token_id = '{token_id}'")
print(f"  {sig0.hex()[:12]}...")
print()
print(f"  sig0           ──HMAC──▶  sig1  | caveat = '{caveat1}'")
print(f"  {sig1.hex()[:12]}...")
print()
print(f"  sig1           ──HMAC──▶  sig2  | caveat = '{caveat2}'")
print(f"  {sig2.hex()[:12]}...")
print()
print("The Guest Token (what the friend carries to the door):")
print(f"  token_id : {token_id}")
print(f"  caveats  : {[caveat1, caveat2]}")
print(f"  signature: {sig2.hex()}")
print("\n✅ Stage 4 Complete: Guest pass forged (legitimately).")

---

## 🔒 Stage 5: The Door Lock — Verification

### Background: How Does the Door Know?

The door reader has access to one thing and one thing only: the **master secret** (it's a trusted server component). When the friend taps their token, the door receives:

- `token_id`: `"Manager-Key-001"`
- `caveats`: `["floor=1", "room=breakroom"]`
- `signature`: `sig2`

The door then **replays the entire chain** from scratch:

```
expected_sig0 = HMAC(master_secret, token_id)
expected_sig1 = HMAC(expected_sig0, "floor=1")
expected_sig2 = HMAC(expected_sig1, "room=breakroom")

if expected_sig2 == provided_sig2:
    ✅ GRANT ACCESS
else:
    ❌ DENY
```

The verifier never needs to know `sig1` or `sig0`. It reconstructs them internally from `master_secret` and the caveat list. Any tampering with the caveat list — adding, removing, or reordering caveats — will produce a different final signature and the door will reject the token.

### ⚠️ Timing Attack Warning

One critical detail: **never use `==` to compare signatures**. Python's `==` short-circuits on the first mismatched byte, which leaks timing information. An attacker making millions of requests can statistically infer the correct signature one byte at a time.

Always use `hmac.compare_digest(a, b)`, which compares all bytes in constant time regardless of where they differ.

### 🛠️ Your Task

Implement `verify_macaroon(master_secret, token_id, caveats, provided_signature)` which:
1. Recomputes the chain from `master_secret` using `mint_macaroon` and `add_caveat`
2. Returns `True` if the recomputed final signature matches `provided_signature`, using `hmac.compare_digest`

In [ ]:
def verify_macaroon(
    master_secret:      bytes,
    token_id:           str,
    caveats:            list,
    provided_signature: bytes
) -> bool:
    """
    Verify a Macaroon token by replaying the full HMAC chain.
    
    The verifier (the door lock) recomputes every signature from the master_secret
    and checks whether the final result matches what the token holder presented.
    
    Args:
        master_secret:      The root secret (only the verifier has this)
        token_id:           The public label from the token
        caveats:            The list of caveats, in the exact order they were applied
        provided_signature: The signature the token holder presented at the door
    
    Returns:
        True if the chain is valid and untampered, False otherwise
    """
    # --- Step 1: Recompute the root signature ---
    # YOUR CODE HERE: use mint_macaroon to get the starting signature
    raise NotImplementedError("Step 1: Recompute root signature")

    # --- Step 2: Walk through the caveats, extending the chain ---
    # YOUR CODE HERE: iterate over caveats and call add_caveat for each
    raise NotImplementedError("Step 2: Replay caveat chain")

    # --- Step 3: Constant-time comparison ---
    # YOUR CODE HERE: return hmac.compare_digest(computed_signature, provided_signature)
    # IMPORTANT: Do NOT use '==' — that's vulnerable to timing attacks!
    raise NotImplementedError("Step 3: Constant-time comparison")

In [ ]:
# --- Test Legitimate Access ---
print("The Guest taps their token at the break room door...")
print()

is_valid = verify_macaroon(
    master_secret      = master_secret,
    token_id           = token_id,
    caveats            = [caveat1, caveat2],
    provided_signature = sig2
)

print(f"Access {'GRANTED ✅' if is_valid else 'DENIED ❌'}")
assert is_valid, "The legitimate token should verify successfully!"
print("\n✅ Stage 5 Complete: Verification system is online.")

---

## 💀 Stage 6: The Attacker Suite — Trying to Break In

### Background: The Building's Enemies

Now you switch hats. You are no longer the friendly security consultant — you're the red team. Three different adversaries each have a copy of the Guest's token:

```
token_id  = 'Manager-Key-001'
caveats   = ['floor=1', 'room=breakroom']
signature = sig2
```

Each adversary has a different plan.

---

### Adversary A: The Deletion Attack

**Goal:** Remove `floor=1` from the caveat list to gain access to all floors.

The adversary hopes the door will compute `HMAC(sig0, "room=breakroom")` instead and that it will match `sig2`.

But think about what the door actually computes:
- With the deletion: `HMAC(sig0, "room=breakroom")` → some new value, call it `sig_fake`
- `sig_fake ≠ sig2`, because `sig2 = HMAC(sig1, "room=breakroom")` and `sig1 ≠ sig0`

The adversary doesn't know `sig0` or `sig1`, so they can't produce the right signature for the modified caveat list.

---

### Adversary B: The Reordering Attack

**Goal:** Swap the order of caveats, hoping the system doesn't care about ordering.

The adversary presents `["room=breakroom", "floor=1"]` with `sig2`.

But the HMAC chain is **ordered** — each signature depends on the previous one. Computing the chain in a different order produces a completely different sequence of signatures, none of which will match `sig2`.

---

### Adversary C: The Forging Attack

**Goal:** Add `admin=true` to the caveats and produce a matching signature.

To produce a valid `sig3 = HMAC(sig2, "admin=true")`, the adversary needs `sig2` as the key. But wait — **the Guest was handed `sig2`**, right?

Actually, no. In a real Macaroon system, the bearer receives the caveat list and the *final* signature, but the door verifier only trusts signatures that *it can reproduce from the master secret*. The Guest can compute `sig3 = HMAC(sig2, "admin=true")` — but when the door re-runs the chain with caveats `["floor=1", "room=breakroom", "admin=true"]`, it will produce the same `sig3`. Does this mean Macaroons are broken?

**Not if the verifier enforces the caveats!** The Macaroon system handles integrity — it proves the chain wasn't tampered with. The **application layer** must then evaluate whether the presented caveats grant the requested action. `admin=true` is only useful if the application blindly trusts it — a properly written verifier will only honour caveats it issued itself.

For this exercise, your forging test simulates a simpler case: the adversary doesn't have `sig2` and tries to present a fake signature. This always fails.

### 🛠️ Your Task

For each attack scenario, call `verify_macaroon` with the tampered inputs and **assert that it returns `False`**. The building's security must hold.

In [ ]:
print("=" * 60)
print("RED TEAM ENGAGEMENT: Testing Attack Scenarios")
print("=" * 60)
print()

# --- Attack A: Deletion --- 
# The adversary removes 'floor=1', hoping to gain building-wide access.
# They still present sig2 as their signature.
print("⚔️  Attack A: Deletion — Remove 'floor=1' from caveats")

# YOUR CODE HERE
# Call verify_macaroon with only [caveat2] (missing caveat1) and sig2
is_valid_deletion = None  # Replace this line
raise NotImplementedError("Run Attack A")

assert not is_valid_deletion, "Deletion attack should be rejected!"
print("   Result: ACCESS DENIED ✅ (Deletion attack failed — chain broken)")
print()

In [ ]:
# --- Attack B: Reordering ---
# The adversary swaps the caveat order, hoping the door doesn't care.
print("⚔️  Attack B: Reordering — Swap caveat order to ['room=breakroom', 'floor=1']")

# YOUR CODE HERE
# Call verify_macaroon with [caveat2, caveat1] (reversed) and sig2
is_valid_reorder = None  # Replace this line
raise NotImplementedError("Run Attack B")

assert not is_valid_reorder, "Reordering attack should be rejected!"
print("   Result: ACCESS DENIED ✅ (Reorder attack failed — signature mismatch)")
print()

In [ ]:
# --- Attack C: Forging ---
# The adversary adds 'admin=true' but doesn't have a valid sig3.
# They construct a bogus signature and hope for the best.
print("⚔️  Attack C: Forging — Add 'admin=true' with a fake signature")

forged_caveat = 'admin=true'
forged_sig    = b'this_is_not_a_real_signature____'  # 32 bytes of nonsense

# YOUR CODE HERE
# Call verify_macaroon with [caveat1, caveat2, forged_caveat] and forged_sig
is_valid_forge = None  # Replace this line
raise NotImplementedError("Run Attack C")

assert not is_valid_forge, "Forging attack should be rejected!"
print("   Result: ACCESS DENIED ✅ (Forge attack failed — can't produce valid sig without sig2)")
print()

In [ ]:
print("=" * 60)
print("All adversarial tests passed.")
print("Galactic Industries HQ is secure.")
print("=" * 60)
print()
print("\n✅ Stage 6 Complete: The building held. Red team engagement finished.")

---

## 🎓 Debrief: What You've Built

Congratulations, Agent. Let's review what you implemented — and why it matters.

### The Macaroon Chain

```
master_secret
    │
    │ HMAC(master_secret, token_id)
    ▼
  sig0  ◀── root token; held by Head of Security
    │
    │ HMAC(sig0, "floor=1")
    ▼
  sig1  ◀── attenuated token; held by Floor Manager
    │
    │ HMAC(sig1, "room=breakroom")
    ▼
  sig2  ◀── guest token; held by the Friend
```

Each level of the hierarchy can **only delegate downward**. You cannot forge a signature for a level you don't have, and you cannot remove constraints once they're embedded.

### Key Properties Demonstrated

| Property | How it's enforced |
|---|---|
| **Unforgeability** | HMAC is a one-way function — you can't reverse a signature |
| **Attenuation-only** | Each level uses the previous sig as a key, so you can't widen permissions |
| **Tamper evidence** | Any change to the caveat list breaks the chain |
| **No secret exposure** | The master secret never appears in any token |
| **Timing safety** | `hmac.compare_digest` prevents side-channel attacks |

### Where Real Macaroons Go Further

Real Macaroon libraries (like [PyMacaroons](https://github.com/ecordell/pymacaroons)) also support:

- **Third-party caveats**: Caveats that must be verified by a *different* service (e.g., "only valid if the auth server confirms this session is active")
- **Discharge tokens**: Tokens issued by third parties to satisfy those caveats
- **Location hints**: Metadata pointing to where a caveat should be verified

These extensions make Macaroons suitable for distributed systems like cloud APIs, where a single backend can't verify all conditions itself.

### Further Reading

- [Macaroons: Cookies with Contextual Caveats (original paper)](https://research.google/pubs/pub41892/)
- [HMAC RFC 2104](https://www.rfc-editor.org/rfc/rfc2104)
- [Length Extension Attack (Wikipedia)](https://en.wikipedia.org/wiki/Length_extension_attack)

---

*The CEO returned from her sailing trip, reviewed the access logs, and approved of everything. The Head of Security got a raise. The Guest enjoyed the break room coffee.*

*Mission complete.* 🍪